# 02 · Gemma — iteración manual del pipeline de texto

En el benchmark inicial se compararon tres familias de modelos abiertas sobre los mismos chunks del corpus: **Qwen3-4B-Instruct-2507, Gemma 3 4B IT y Mistral 7B Instruct v0.3**.

Para continuar el prototipado se selecciona **Gemma 3 4B IT** porque, en la muestra inicial:

- produjo salida JSON válida de forma consistente;
- mostró buena adherencia al texto fuente;
- tuvo una latencia competitiva;
- con 4B parámetros resulta razonable para iterar en Google Colab.

Esta decisión es **pragmática y provisional**: el objetivo de este notebook no es establecer un benchmark exhaustivo, sino depurar manualmente el comportamiento del pipeline antes de automatizarlo.

## Dos estrategias que vamos a comparar

### Pipeline 1 — Extract → Select → Write
`chunk → hasta 3 ideas → selección de la idea relevante → texto final corto`

Esta estrategia fuerza al modelo a separar extracción factual y redacción.

### Pipeline 2 — Direct summary
`chunk completo → texto final de ~30 segundos`

Esta estrategia sirve como baseline simple para comprobar si la descomposición del Pipeline 1 aporta valor.

In [1]:
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes huggingface_hub pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 31.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


## 1. Clonar FinTok y cargar el corpus

In [2]:
from pathlib import Path
import json, random, gc, re, time
import numpy as np
import pandas as pd
import torch

# Clonar solo si todavía no existe en esta sesión de Colab
if not Path("/content/FinTok").exists():
    !git clone https://github.com/mirdbg/FinTok.git

REPO_DIR = Path("/content/FinTok")
ZIP_PATH = REPO_DIR / "data" / "corpus_miax_2026.zip"
WORK_DIR = Path("/content/corpus_miax_2026")

WORK_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"No encuentro: {ZIP_PATH}"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Corpus:", ZIP_PATH)

Cloning into 'FinTok'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 27 (delta 4), reused 18 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 1.76 MiB | 23.65 MiB/s, done.
Resolving deltas: 100% (4/4), done.
GPU: Tesla T4
Corpus: /content/FinTok/data/corpus_miax_2026.zip


In [3]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(WORK_DIR)

with open(WORK_DIR / "chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(chunks)

print(f"Chunks cargados: {len(df):,}")
display(df[["chunk_id", "ticker", "fiscal_year", "item", "n_tokens", "contiene_tabla"]].head())

Chunks cargados: 1,749


,chunk_id,ticker,fiscal_year,item,n_tokens,contiene_tabla
0,NVDA-2024-1A-0000,NVDA,2024,1A,475,False
1,NVDA-2024-1A-0001,NVDA,2024,1A,278,False
2,NVDA-2024-1A-0002,NVDA,2024,1A,500,False
3,NVDA-2024-1A-0003,NVDA,2024,1A,500,False
4,NVDA-2024-1A-0004,NVDA,2024,1A,99,False


## 2. Cargar Gemma

El notebook usa cuantización 4-bit para reducir VRAM.

`HF_TOKEN` debe estar guardado en **Secrets** de Colab y tener activado *Acceso desde el cuaderno*.

In [4]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
print("Hugging Face autenticado.")

Hugging Face autenticado.


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-4b-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=quant_config,
)

model.eval()
print("Modelo cargado:", MODEL_ID)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Modelo cargado: google/gemma-3-4b-it


## 3. Función común de generación

In [6]:
def ask_gemma(system_prompt, user_prompt, max_new_tokens=600):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start
    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(generated, skip_special_tokens=True).strip(), elapsed

## 4. Elegir manualmente un chunk

Durante esta fase queremos **iterar manualmente**, así que no hacemos todavía un bucle por los 1.749 chunks.

Puedes filtrar el DataFrame, leer candidatos y cambiar `CHUNK_ID` tantas veces como quieras.

In [7]:
# Algunos candidatos para explorar
display(
    df.sample(15, random_state=42)[
        ["chunk_id", "ticker", "fiscal_year", "item", "n_tokens", "contiene_tabla"]
    ]
)

,chunk_id,ticker,fiscal_year,item,n_tokens,contiene_tabla
660,AAPL-2024-8-0043,AAPL,2024,8,278,False
1353,META-2025-7-0010,META,2025,7,500,False
861,GOOGL-2024-8-0037,GOOGL,2024,8,370,True
111,NVDA-2024-8-0035,NVDA,2024,8,453,True
332,MSFT-2024-7-0001,MSFT,2024,7,500,False
1005,GOOGL-2025-8-0026,GOOGL,2025,8,500,True
1165,META-2024-7-0020,META,2024,7,157,True
109,NVDA-2024-8-0033,NVDA,2024,8,484,True
135,NVDA-2024-8-0059,NVDA,2024,8,470,False
1440,META-2025-8-0058,META,2025,8,500,False


In [8]:
# CAMBIA ESTE ID para probar otro chunk
CHUNK_ID = df.sample(1, random_state=42).iloc[0]["chunk_id"]

row = df.loc[df["chunk_id"] == CHUNK_ID].iloc[0]
chunk_text = row["texto"]

print("CHUNK:", row["chunk_id"])
print("TICKER:", row["ticker"])
print("AÑO:", row["fiscal_year"])
print("ITEM:", row["item"])
print("=" * 100)
print(chunk_text)

CHUNK: AAPL-2024-8-0043
TICKER: AAPL
AÑO: 2024
ITEM: 8
Definition and Limitations of Internal Control Over Financial Reporting

A company’s internal control over financial reporting is a process designed to provide reasonable assurance regarding the reliability of financial reporting and the preparation of financial statements for external purposes in accordance with generally accepted accounting principles. A company’s internal control over financial reporting includes those policies and procedures that (1) pertain to the maintenance of records that, in reasonable detail, accurately and fairly reflect the transactions and dispositions of the assets of the company; (2) provide reasonable assurance that transactions are recorded as necessary to permit preparation of financial statements in accordance with generally accepted accounting principles, and that receipts and expenditures of the company are being made only in accordance with authorizations of management and directors of the com

# PIPELINE 1 · Extract → Select → Write

Esta celda realiza deliberadamente **tres operaciones dentro de una única llamada**:

1. Extrae hasta tres ideas factuales de 3 chunks consecutivos.
2. Decide si alguna tiene suficiente valor para FinTok.
3. Si existe una idea adecuada, redacta un texto corto y autosuficiente.

El modelo puede devolver `DISCARD`. No queremos obligarlo a fabricar contenido a partir de un fragmento irrelevante.

Durante la iteración manual pedimos también que muestre las ideas candidatas y la razón de selección. Más adelante podremos ocultar esa traza y conservar únicamente `final_text`.

In [11]:
display(
    df[
        ["chunk_id", "ticker", "fiscal_year", "item", "posicion", "n_tokens"]
    ].sample(5, random_state=42)
)

,chunk_id,ticker,fiscal_year,item,posicion,n_tokens
660,AAPL-2024-8-0043,AAPL,2024,8,43,278
1353,META-2025-7-0010,META,2025,7,10,500
861,GOOGL-2024-8-0037,GOOGL,2024,8,37,370
111,NVDA-2024-8-0035,NVDA,2024,8,35,453
332,MSFT-2024-7-0001,MSFT,2024,7,1,500


In [ ]:
'''
Chunks probados
"AAPL-2025-7-0009"
"AAPL-2024-8-0043"
'''

### PASO 1: EXTRAER

In [24]:
# Elegimos manualmente el chunk CENTRAL y añadimos los adyacentes

CHUNK_ID = "AAPL-2024-8-0043"  # cambia aquí el chunk que quieras probar

target = df.loc[df["chunk_id"] == CHUNK_ID].iloc[0]

# Nos quedamos en la misma empresa, año e Item
section_df = (
    df[
        (df["ticker"] == target["ticker"]) &
        (df["fiscal_year"] == target["fiscal_year"]) &
        (df["item"] == target["item"])
    ]
    .sort_values("posicion")
    .reset_index(drop=True)
)

# Localizamos el chunk elegido
center_idx = section_df.index[
    section_df["chunk_id"] == CHUNK_ID
][0]

# Chunk anterior + central + posterior
start = max(0, center_idx - 1)
end = min(len(section_df), center_idx + 2)

context_df = section_df.iloc[start:end]

display(
    context_df[
        ["chunk_id", "ticker", "fiscal_year", "item", "posicion"]
    ]
)

context_text = "\n\n".join(
    f"""--- CHUNK {row['chunk_id']} ---
{row['texto']}"""
    for _, row in context_df.iterrows()
)

print(context_text)

,chunk_id,ticker,fiscal_year,item,posicion
42,AAPL-2024-8-0042,AAPL,2024,8,42
43,AAPL-2024-8-0043,AAPL,2024,8,43


--- CHUNK AAPL-2024-8-0042 ---
Opinion on Internal Control Over Financial Reporting

We have audited Apple Inc.’s internal control over financial reporting as of September 28, 2024, based on criteria established in Internal Control – Integrated Framework issued by the Committee of Sponsoring Organizations of the Treadway Commission (2013 framework) (the “COSO criteria”). In our opinion, Apple Inc. (the “Company”) maintained, in all material respects, effective internal control over financial reporting as of September 28, 2024, based on the COSO criteria.

We also have audited, in accordance with the standards of the Public Company Accounting Oversight Board (United States) (“PCAOB”), the consolidated balance sheets of the Company as of September 28, 2024 and September 30, 2023, the related consolidated statements of operations, comprehensive income, shareholders’ equity and cash flows for each of the three years in the period ended September 28, 2024, and the related notes and our repo

In [25]:
EXTRACT_SYSTEM = """
You are a financial analyst specialized in SEC 10-K filings.

Read the supplied excerpts from three consecutive chunks of the same 10-K section.

Extract up to 3 distinct, self-contained factual ideas that are explicitly
supported by the text.

Prioritize:
- important quantitative results or changes,
- year-over-year developments,
- business drivers,
- strategic developments,
- customer or supplier dependencies,
- material risks,
- unusual or potentially meaningful facts.

Rules:
- Use ONLY information contained in the supplied excerpts.
- Do not use external knowledge.
- Do not speculate.
- Do not exaggerate.
- Combine information across the three chunks when they clearly belong to
  the same idea.
- Do not create separate ideas just because the same information appears
  across overlapping chunks.
- Return fewer than 3 ideas if necessary.
- If nothing meaningful exists, return an empty list.
- This is factual extraction, NOT social-media copywriting.

Return ONLY valid JSON:

{
  "ideas": [
    {
      "idea": "...",
      "evidence": "...",
      "source_chunks": ["chunk_id"]
    }
  ]
}
""".strip()


extract_user = f"""
COMPANY/TICKER: {target['ticker']}
FISCAL YEAR: {target['fiscal_year']}
10-K ITEM: {target['item']}

CONSECUTIVE 10-K EXCERPTS:
----------------
{context_text}
----------------
""".strip()


extract_output, extract_latency = ask_gemma(
    EXTRACT_SYSTEM,
    extract_user,
    max_new_tokens=550
)

print(f"Latencia: {extract_latency:.2f}s")
print(extract_output)

Latencia: 47.34s
```json
{
  "ideas": [
    {
      "idea": "Apple Inc. conducted an audit of its internal control over financial reporting as of September 28, 2024 and September 30, 2023.",
      "evidence": "The auditor audited Apple Inc.’s internal control over financial reporting as of September 28, 2024, based on criteria established in Internal Control – Integrated Framework issued by the Committee of Sponsoring Organizations of the Treadway Commission (2013 framework). We also have audited, in accordance with the standards of the Public Company Accounting Oversight Board (United States) (“PCAOB”), the consolidated balance sheets of the Company as of September 28, 2024 and September 30, 2023, the related consolidated statements of operations, comprehensive income, shareholders’ equity and cash flows for each of the three years in the period ended September 28, 2024.",
      "source_chunks": ["AAPL-2024-8-0042", "AAPL-2024-8-0043"]
    },
    {
      "idea": "The audit was perform

### PASO 2: SELECCIONAR

In [26]:
SELECT_SYSTEM = """
You are the financial editor of FinTok.

Your task is to decide whether any of the candidate ideas extracted from a
10-K is genuinely worth communicating to a general but financially interested
audience.

A fact being correct does NOT automatically make it interesting.

Good candidates tend to involve:
- substantial changes in the business,
- striking financial figures,
- important growth or declines,
- meaningful risks,
- important dependencies,
- strategic changes,
- information that reveals something useful about how the company operates.

Reject ideas that are mainly:
- administrative,
- routine accounting language,
- legal boilerplate,
- too technical without broader relevance,
- impossible to understand without extensive context,
- repetitive or trivial.

Be selective.

Select AT MOST ONE idea.

Use the original excerpts to verify that the selected idea is properly
supported.

Return ONLY valid JSON:

{
  "decision": "KEEP or DISCARD",
  "selected_idea": "..." or null,
  "reason": "..."
}
""".strip()


select_user = f"""
ORIGINAL 10-K EXCERPTS:
----------------
{context_text}
----------------

CANDIDATE IDEAS:
----------------
{extract_output}
----------------
""".strip()


select_output, select_latency = ask_gemma(
    SELECT_SYSTEM,
    select_user,
    max_new_tokens=300
)

print(f"Latencia: {select_latency:.2f}s")
print(select_output)

Latencia: 17.63s
```json
{
  "decision": "KEEP",
  "selected_idea": "Internal control over financial reporting may not prevent or detect misstatements due to inherent limitations and potential changes in conditions.",
  "reason": "This idea highlights a crucial risk factor for investors. While the audit confirms a reasonable level of control, it also explicitly states that controls can fail and misstatements can occur. This is a significant caveat that investors need to understand, and it’s a more insightful piece of information than simply stating that an audit was performed or that the company is audited by a specific firm. It reveals a potential vulnerability and the inherent difficulty in achieving perfect accuracy in financial reporting."
}
```


### PASO 3: ESCRIBIR

In [35]:
# ============================================================
# FASE 3 — WRITE
# Generación del hook + explicación + highlights visuales
# ============================================================

WRITE_SYSTEM = """
You are a financial content writer for FinTok, a platform that transforms
official 10-K filings into short, engaging and easy-to-understand financial content.

Turn the selected financial idea into a natural spoken script for a short video.

Use the original 10-K excerpts as the ONLY factual source.

Your goal is to communicate ONE interesting financial story clearly and naturally.


1. HOOK

Write ONE short and engaging question or surprising factual statement based
on the selected idea.

The hook should make the viewer curious and want to keep watching.

IMPORTANT:
- The hook must contain ONLY the hook.
- Do not explain the full story in the hook.
- Maximum 25 words.
- Make it specific to the selected financial story.

You may use formats such as:
- "Did you know that...?"
- "What if I told you that...?"
- "How much...?"
- "Why did...?"
- "One number in [company]'s latest filing stands out..."

Do NOT use misleading clickbait.
Do NOT exaggerate beyond what is supported by the source.


HOOK HIGHLIGHT

Also create a "hook_highlight".

This is a very short visual summary of the hook that may appear as large
on-screen text.

Rules:
- Ideally 3-6 words.
- Preserve the main idea of the hook.
- It may be a question or statement.
- It does NOT need to contain a number.
- Do not introduce new information.


2. EXPLANATION

Explain the selected idea clearly and naturally.

Write 2-3 concise sentences.

IMPORTANT:
- Maximum 70 words.
- Focus on ONE main financial story.
- Every sentence must add NEW information or context.
- Do not repeat the same idea in different words.
- Stop once the main idea has been explained.

The explanation must:
- identify the company naturally;
- preserve the most relevant figures and context;
- explain the fact in language understandable to a general interested audience;
- sound natural when spoken aloud;
- avoid unnecessary financial jargon;
- avoid simply copying regulatory language;
- avoid investment advice;
- avoid unsupported interpretations;
- use ONLY information supported by the supplied 10-K excerpts;
- simplify regulatory language into natural spoken English while preserving
  its factual meaning.

Do not try to summarize everything in the excerpts.

Only include information that directly helps explain the selected story.


EXPLANATION HIGHLIGHT

Also create an "explanation_highlight".

This is ONE short relevant fact from the explanation that may appear as
large on-screen text.

It may be:

- quantitative: a monetary amount, percentage, growth rate, ratio or
  other important numerical fact;

OR

- qualitative: an important business fact, risk, dependency, strategic
  development or limitation.

Rules:
- Ideally 3-7 words.
- Choose the most relevant or interesting fact from the explanation.
- Prefer a quantitative fact when an important number is central to the story.
- Otherwise use a concise qualitative fact.
- It must refer to information already contained in the explanation.
- Never introduce a new fact.


FINTOK INTRO AND OUTRO

The following two sentences will be added automatically later by Python:

"This is FinTok — your daily dose of financial knowledge."

"That's your FinTok for today. See you in the next one."

Therefore:
- DO NOT generate the FinTok intro.
- DO NOT generate the FinTok outro.


OUTPUT

Return ONLY valid JSON with EXACTLY this structure:

{
  "hook": "...",
  "hook_highlight": "...",
  "explanation": "...",
  "explanation_highlight": "..."
}

Do not include markdown.
Do not include ```json.
Do not include any text before or after the JSON.
""".strip()


write_user = f"""
SELECTED FINANCIAL STORY:
----------------
{select_output}
----------------

SOURCE 10-K EXCERPTS:
----------------
{context_text}
----------------

Write the hook, explanation and their visual highlights.

Remember:
- hook: maximum 25 words;
- explanation: maximum 70 words;
- use only information supported by the source.
""".strip()


write_output, write_latency = ask_gemma(
    WRITE_SYSTEM,
    write_user,
    max_new_tokens=350
)

print(f"Latencia: {write_latency:.2f}s")
print()
print(write_output)

Latencia: 18.55s

```json
{
  "hook": "Did you know that Apple’s latest filing admits their internal controls aren’t foolproof, potentially leading to misstatements?",
  "hook_highlight": "Internal controls aren’t perfect.",
  "explanation": "Apple Inc. states that its internal controls over financial reporting are designed to ensure accurate financial statements. However, the company acknowledges inherent limitations – controls can’t always prevent or detect misstatements. Furthermore, projections of control effectiveness are subject to change due to evolving conditions, highlighting a potential vulnerability in their reporting process.",
  "explanation_highlight": "Controls may not prevent misstatements."
}
```


### PASO 4: FORMATEAMOS A JSON

In [39]:
# ============================================================
# FASE 4 — BUILD FINAL VIDEO JSON
# Construcción determinista del JSON final de FinTok
# ============================================================

import json
import re


# ------------------------------------------------------------
# 1. Parsear la salida de WRITE
# ------------------------------------------------------------

def parse_json_output(text):
    """
    Convierte la salida de Gemma en un diccionario Python.
    También elimina bloques ```json si el modelo los añade.
    """
    text = text.strip()

    # Eliminar posibles markdown fences
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    return json.loads(text)


write_data = parse_json_output(write_output)


# ------------------------------------------------------------
# 2. Validar la salida de WRITE
# ------------------------------------------------------------

required_fields = {
    "hook",
    "hook_highlight",
    "explanation",
    "explanation_highlight"
}

missing_fields = required_fields - set(write_data.keys())

if missing_fields:
    raise ValueError(
        f"WRITE output is missing required fields: {missing_fields}"
    )


# ------------------------------------------------------------
# 3. Elementos fijos de FinTok
# ------------------------------------------------------------

INTRO = "This is FinTok — your daily dose of financial knowledge."

OUTRO = "That's your FinTok for today. See you in the next one."


# ------------------------------------------------------------
# 4. Provenance
# ------------------------------------------------------------

# Los tres chunks consecutivos utilizados como contexto
# durante EXTRACT / SELECT / WRITE.
source_chunk_ids = context_df["chunk_id"].tolist()


# ------------------------------------------------------------
# 5. Construir el JSON final
# ------------------------------------------------------------

video_data = {
    "company": str(target["ticker"]),
    "year": int(target["fiscal_year"]),
    "language": "en",

    "scenes": [
        {
            "scene_id": 1,
            "scene_type": "hook",
            "speaker": None,
            "text": write_data["hook"],
            "key_figure": write_data["hook_highlight"],
            "source_chunk_ids": source_chunk_ids,
            "verified": True,
            "audio_path": None
        },

        {
            "scene_id": 2,
            "scene_type": "intro",
            "speaker": None,
            "text": INTRO,
            "key_figure": None,
            "source_chunk_ids": [],
            "verified": True,
            "audio_path": None
        },

        {
            "scene_id": 3,
            "scene_type": "explanation",
            "speaker": None,
            "text": write_data["explanation"],
            "key_figure": write_data["explanation_highlight"],
            "source_chunk_ids": source_chunk_ids,
            "verified": True,
            "audio_path": None
        },

        {
            "scene_id": 4,
            "scene_type": "outro",
            "speaker": None,
            "text": OUTRO,
            "key_figure": None,
            "source_chunk_ids": [],
            "verified": True,
            "audio_path": None
        }
    ]
}


# ------------------------------------------------------------
# 6. Validaciones
# ------------------------------------------------------------

assert len(video_data["scenes"]) == 4

assert [scene["scene_type"] for scene in video_data["scenes"]] == [
    "hook",
    "intro",
    "explanation",
    "outro"
]

# Intro y outro deben ser exactamente los definidos por FinTok
assert video_data["scenes"][1]["text"] == INTRO
assert video_data["scenes"][3]["text"] == OUTRO

# Intro y outro no tienen highlight
assert video_data["scenes"][1]["key_figure"] is None
assert video_data["scenes"][3]["key_figure"] is None

# Intro y outro no necesitan fuentes
assert video_data["scenes"][1]["source_chunk_ids"] == []
assert video_data["scenes"][3]["source_chunk_ids"] == []

# Todavía no asignamos voces ni audios
for scene in video_data["scenes"]:
    assert scene["speaker"] is None
    assert scene["audio_path"] is None

# Comprobar que todos los source chunks son reales
available_chunk_ids = set(context_df["chunk_id"])

for scene in video_data["scenes"]:
    for chunk_id in scene["source_chunk_ids"]:
        assert chunk_id in available_chunk_ids, (
            f"Unknown source chunk: {chunk_id}"
        )


# ------------------------------------------------------------
# 7. Mostrar JSON final
# ------------------------------------------------------------


print(
    json.dumps(
        video_data,
        indent=2,
        ensure_ascii=False
    )
)


{
  "company": "AAPL",
  "year": 2024,
  "language": "en",
  "scenes": [
    {
      "scene_id": 1,
      "scene_type": "hook",
      "speaker": null,
      "text": "Did you know that Apple’s latest filing admits their internal controls aren’t foolproof, potentially leading to misstatements?",
      "key_figure": "Internal controls aren’t perfect.",
      "source_chunk_ids": [
        "AAPL-2024-8-0042",
        "AAPL-2024-8-0043"
      ],
      "verified": true,
      "audio_path": null
    },
    {
      "scene_id": 2,
      "scene_type": "intro",
      "speaker": null,
      "text": "This is FinTok — your daily dose of financial knowledge.",
      "key_figure": null,
      "source_chunk_ids": [],
      "verified": true,
      "audio_path": null
    },
    {
      "scene_id": 3,
      "scene_type": "explanation",
      "speaker": null,
      "text": "Apple Inc. states that its internal controls over financial reporting are designed to ensure accurate financial statements. However, th

## Próximos pasos

Cuando el comportamiento sea satisfactorio, este notebook se puede convertir en un pipeline automático:

`todos los chunks → Pipeline 1 → KEEP → generación del guion definitivo → voz / imagen / vídeo`

Antes de automatizarlo conviene decidir manualmente:
1. qué entendemos por una idea suficientemente interesante;
2. cuánto debe durar el contenido;
3. qué tono editorial queremos;
4. si una sola idea por vídeo es la estrategia adecuada.